# Saddle Point Detection & Critical Structure Analysis

Comprehensive analysis of critical points in vector fields:
- **Hyperbolic structures** (index-1 saddles): λ₊ > 0, λ₋ < 0
- **Elliptic structures** (ridges/separatrix): λ₊ < 0, λ₋ < 0
- **Scalar field classification**: Degenerate, saddle, min/max points
- **Field visualization**: 4-panel comprehensive analysis

Adjust hyperparameters and run each section independently.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

## Vector Field Definition

In [ ]:
class VectorField:
    """Four field types: 'double_vortex', 'double_saddle', 'double_gyre', 'four_vortex'."""

    def __init__(self, field_type='double_vortex',
                 dg_A=0.1, dg_eps=0.0, dg_omega=2*np.pi/10, dg_t=0.0):
        self.field_type = field_type
        self.vortex1_pos = np.array([-2, -1])
        self.vortex2_pos = np.array([2, -1])
        self.saddle1_pos = np.array([3, 3])
        self.saddle2_pos = np.array([-3, -3])
        self.fv_centers = np.array([[-1,3],[1,3],[-1,-1],[1,-1]], dtype=float)
        self.fv_signs   = [1, -1, -1, 1]
        self.dg_A = dg_A
        self.dg_eps = dg_eps
        self.dg_omega = dg_omega
        self.dg_t = dg_t

    def evaluate(self, x, y):
        if self.field_type == 'double_vortex':
            x1, y1 = self.vortex1_pos
            vx1 = -(y - y1); vy1 = (x - x1)
            x2, y2 = self.vortex2_pos
            vx2 =  (y - y2); vy2 = -(x - x2)
            d1 = np.sqrt((x - x1)**2 + (y - y1)**2)
            d2 = np.sqrt((x - x2)**2 + (y - y2)**2)
            w1 = 1.0 / (d1**2 + 0.1)
            w2 = 1.0 / (d2**2 + 0.1)
            vx = (vx1 * w1 + vx2 * w2) / (w1 + w2)
            vy = (vy1 * w1 + vy2 * w2) / (w1 + w2)
            return np.array([10*vx, 10*vy])

        elif self.field_type == 'double_saddle':
            s1 = self.saddle1_pos; s2 = self.saddle2_pos
            rot1, rot2 = np.pi/4, -np.pi/4
            x1c = x - s1[0]; y1c = y - s1[1]
            c1, si1 = np.cos(rot1), np.sin(rot1)
            x1r = c1*x1c - si1*y1c; y1r = si1*x1c + c1*y1c
            u1r =  x1r; v1r = -y1r
            u1 =  c1*u1r + si1*v1r
            v1 = -si1*u1r +  c1*v1r
            x2c = x - s2[0]; y2c = y - s2[1]
            c2, si2 = np.cos(rot2), np.sin(rot2)
            x2r = c2*x2c - si2*y2c; y2r = si2*x2c + c2*y2c
            u2r =  x2r; v2r = -y2r
            u2 =  c2*u2r + si2*v2r
            v2 = -si2*u2r +  c2*v2r
            d1 = np.sqrt((x - s1[0])**2 + (y - s1[1])**2) + 0.01
            d2 = np.sqrt((x - s2[0])**2 + (y - s2[1])**2) + 0.01
            w1 = 1.0 / (d1**2); w2 = 1.0 / (d2**2)
            tw = w1 + w2
            return np.array([(u1*w1 + u2*w2)/tw, (v1*w1 + v2*w2)/tw])

        elif self.field_type == 'double_gyre':
            A = self.dg_A; eps = self.dg_eps
            omega = self.dg_omega; t = self.dg_t
            f  = eps * np.sin(omega*t) * x**2 + (1 - 2*eps*np.sin(omega*t)) * x
            df = 2*eps * np.sin(omega*t) * x + (1 - 2*eps*np.sin(omega*t))
            u = -np.pi * A * np.sin(np.pi*f) * np.cos(np.pi*y)
            v =  np.pi * A * np.cos(np.pi*f) * np.sin(np.pi*y) * df
            return np.array([u, v])

        elif self.field_type == 'four_vortex':
            eps = 0.1
            vx, vy = 0.0, 0.0
            for (cx, cy), sign in zip(self.fv_centers, self.fv_signs):
                dx, dy = x - cx, y - cy
                r2 = dx**2 + dy**2 + eps
                vx += -sign * dy / r2
                vy +=  sign * dx / r2
            return np.array([vx, vy])

    def jacobian_twosided(self, x, y, h=0.01):
        fxp, fyp = self.evaluate(x + h, y)
        fxm, fym = self.evaluate(x - h, y)
        fxyp, fyyp = self.evaluate(x, y + h)
        fxym, fyym = self.evaluate(x, y - h)
        return np.array([
            [(fxp - fxm)/(2*h), (fxyp - fxym)/(2*h)],
            [(fyp - fym)/(2*h), (fyyp - fyym)/(2*h)],
        ])

## Computation Functions

In [ ]:
def compute_critical_structure_fields(field, x_range, y_range, grid_res=80):
    """Compute det(J), gradients, and Hessian eigenvalues on a grid."""
    xs = np.linspace(x_range[0], x_range[1], grid_res)
    ys = np.linspace(y_range[0], y_range[1], grid_res)
    X, Y = np.meshgrid(xs, ys)
    
    det_grid = np.zeros_like(X)
    grad_norm = np.zeros_like(X)
    lambda_max = np.zeros_like(X)
    lambda_min = np.zeros_like(X)
    
    h = (x_range[1] - x_range[0]) / (grid_res - 1) * 2
    
    for i in range(grid_res):
        for j in range(grid_res):
            x, y = X[i, j], Y[i, j]
            
            J = field.jacobian_twosided(x, y, h=h/2)
            det_J = np.linalg.det(J)
            det_grid[i, j] = det_J
            
            J_xp = field.jacobian_twosided(x + h, y, h=h/2)
            J_xm = field.jacobian_twosided(x - h, y, h=h/2)
            J_yp = field.jacobian_twosided(x, y + h, h=h/2)
            J_ym = field.jacobian_twosided(x, y - h, h=h/2)
            
            grad_x = (np.linalg.det(J_xp) - np.linalg.det(J_xm)) / (2*h)
            grad_y = (np.linalg.det(J_yp) - np.linalg.det(J_ym)) / (2*h)
            grad_norm[i, j] = np.sqrt(grad_x**2 + grad_y**2)
            
            ddet_xp_yp = np.linalg.det(field.jacobian_twosided(x + h, y + h, h=h/2))
            ddet_xp_ym = np.linalg.det(field.jacobian_twosided(x + h, y - h, h=h/2))
            ddet_xm_yp = np.linalg.det(field.jacobian_twosided(x - h, y + h, h=h/2))
            ddet_xm_ym = np.linalg.det(field.jacobian_twosided(x - h, y - h, h=h/2))
            
            H_xx = (ddet_xp_yp - 2*det_J + ddet_xm_ym) / (h**2)
            H_yy = (ddet_xp_yp - 2*det_J + ddet_xm_ym) / (h**2)
            H_xy = (ddet_xp_yp - ddet_xp_ym - ddet_xm_yp + ddet_xm_ym) / (4*h**2)
            
            H = np.array([[H_xx, H_xy], [H_xy, H_yy]])
            evals = np.linalg.eigvalsh(H)
            lambda_max[i, j] = evals[1]
            lambda_min[i, j] = evals[0]
    
    return det_grid, grad_norm, lambda_max, lambda_min


def compute_detection_masks(det_grid, grad_norm, lambda_max, lambda_min,
                            eps_g, eps_lambda, eps_det):
    """Compute masks for saddle and ridge detection."""
    saddle_mask = (grad_norm < eps_g) & (lambda_max > eps_lambda) & (lambda_min < -eps_lambda)
    ridge_mask = (np.abs(det_grid) < eps_det) & (grad_norm < eps_g) & \
                 (lambda_max < -eps_lambda) & (lambda_min < -eps_lambda)
    return saddle_mask, ridge_mask


def classify_critical_points(grad_norm, det_H, lambda_1, lambda_2, eps_1, eps_2):
    """Classify critical points: degenerate, saddle, or min/max."""
    is_critical = grad_norm < eps_1
    is_degenerate = np.abs(det_H) < eps_2
    
    degenerate_mask = is_critical & is_degenerate
    non_degen = is_critical & ~is_degenerate
    saddle_mask = non_degen & (lambda_1 * lambda_2 < 0)
    minmax_mask = non_degen & (lambda_1 * lambda_2 > 0)
    
    return degenerate_mask, saddle_mask, minmax_mask

# Visualization 1: Extended Critical Point Detection (Hyperbolic & Elliptic)

In [ ]:
# Hyperparameters for Visualization 1
field_type_1 = 'double_gyre'  # 'double_vortex', 'double_saddle', 'double_gyre', 'four_vortex'
eps_g_1 = 1.0                 # gradient tolerance for critical point
eps_lambda_1 = 0.3            # eigenvalue magnitude threshold
eps_det_1 = 0.15              # determinant tolerance for ridge detection

# Domain selection
if field_type_1 == 'double_gyre':
    x_range, y_range = [0, 2], [0, 1]
    grid_res = 80
else:
    x_range, y_range = [-5, 5], [-5, 5]
    grid_res = 80

field = VectorField(field_type=field_type_1)

xs = np.linspace(x_range[0], x_range[1], grid_res)
ys = np.linspace(y_range[0], y_range[1], grid_res)
X, Y = np.meshgrid(xs, ys)

det_grid, grad_norm, lambda_max, lambda_min = compute_critical_structure_fields(
    field, x_range, y_range, grid_res=grid_res)

saddle_mask, ridge_mask = compute_detection_masks(
    det_grid, grad_norm, lambda_max, lambda_min, eps_g_1, eps_lambda_1, eps_det_1)

# Create figure
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'{field_type_1} | ε_g={eps_g_1:.3f}, ε_λ={eps_lambda_1:.3f}, ε_det={eps_det_1:.3f}',
             fontsize=12, fontweight='bold')

# Saddle detection (red)
ax = axes[0]
ax.contourf(X, Y, det_grid, levels=20, cmap='RdBu_r', alpha=0.4)
ax.contour(X, Y, det_grid, levels=[0], colors='gray', linewidths=1, linestyles='--')
ax.contourf(X, Y, saddle_mask.astype(float), levels=[0.5, 1.5], colors=['red'], alpha=0.6)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title(f'SADDLE Detection\n(λ₊ > {eps_lambda_1:.3f}, λ₋ < {-eps_lambda_1:.3f})')
ax.set_aspect('equal')
saddle_count = np.sum(saddle_mask)
ax.text(0.02, 0.98, f'{saddle_count} points', transform=ax.transAxes,
        fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='red', alpha=0.3))

# Ridge detection (blue)
ax = axes[1]
ax.contourf(X, Y, det_grid, levels=20, cmap='RdBu_r', alpha=0.4)
ax.contour(X, Y, det_grid, levels=[0], colors='gray', linewidths=1, linestyles='--')
ax.contourf(X, Y, ridge_mask.astype(float), levels=[0.5, 1.5], colors=['blue'], alpha=0.6)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title(f'RIDGE Detection\n(|det(J)| < {eps_det_1:.3f}, λ₊,λ₋ < {-eps_lambda_1:.3f})')
ax.set_aspect('equal')
ridge_count = np.sum(ridge_mask)
ax.text(0.02, 0.98, f'{ridge_count} points', transform=ax.transAxes,
        fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='blue', alpha=0.3))

# Combined view
ax = axes[2]
ax.contourf(X, Y, det_grid, levels=20, cmap='RdBu_r', alpha=0.4)
ax.contour(X, Y, det_grid, levels=[0], colors='black', linewidths=2)
ax.contourf(X, Y, saddle_mask.astype(float), levels=[0.5, 1.5], colors=['red'], alpha=0.5)
ax.contourf(X, Y, ridge_mask.astype(float), levels=[0.5, 1.5], colors=['blue'], alpha=0.5)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('COMBINED Detection')
ax.set_aspect('equal')
ax.text(0.02, 0.98, f'Red (saddle): {saddle_count}\nBlue (ridge): {ridge_count}', 
        transform=ax.transAxes, fontsize=9, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

# Visualization 2: Scalar Field Classification (Degenerate, Saddle, Min/Max)

In [ ]:
def compute_scalar_field_analysis(field, x_range, y_range, grid_res=80):
    """Compute det(J), grad norm, and Hessian eigenvalues."""
    xs = np.linspace(x_range[0], x_range[1], grid_res)
    ys = np.linspace(y_range[0], y_range[1], grid_res)
    X, Y = np.meshgrid(xs, ys)
    
    det_J_grid = np.zeros_like(X)
    grad_norm = np.zeros_like(X)
    lambda_1 = np.zeros_like(X)
    lambda_2 = np.zeros_like(X)
    
    h = (x_range[1] - x_range[0]) / (grid_res - 1) * 2
    
    for i in range(grid_res):
        for j in range(grid_res):
            x, y = X[i, j], Y[i, j]
            
            J = field.jacobian_twosided(x, y, h=h/2)
            det_J = np.linalg.det(J)
            det_J_grid[i, j] = det_J
            
            J_xp = field.jacobian_twosided(x + h, y, h=h/2)
            J_xm = field.jacobian_twosided(x - h, y, h=h/2)
            J_yp = field.jacobian_twosided(x, y + h, h=h/2)
            J_ym = field.jacobian_twosided(x, y - h, h=h/2)
            
            grad_x = (np.linalg.det(J_xp) - np.linalg.det(J_xm)) / (2*h)
            grad_y = (np.linalg.det(J_yp) - np.linalg.det(J_ym)) / (2*h)
            grad_norm[i, j] = np.sqrt(grad_x**2 + grad_y**2)
            
            ddet_xp_yp = np.linalg.det(field.jacobian_twosided(x + h, y + h, h=h/2))
            ddet_xp_ym = np.linalg.det(field.jacobian_twosided(x + h, y - h, h=h/2))
            ddet_xm_yp = np.linalg.det(field.jacobian_twosided(x - h, y + h, h=h/2))
            ddet_xm_ym = np.linalg.det(field.jacobian_twosided(x - h, y - h, h=h/2))
            
            H_xx = (ddet_xp_yp - 2*det_J + ddet_xm_ym) / (h**2)
            H_yy = (ddet_xp_yp - 2*det_J + ddet_xm_ym) / (h**2)
            H_xy = (ddet_xp_yp - ddet_xp_ym - ddet_xm_yp + ddet_xm_ym) / (4*h**2)
            
            H = np.array([[H_xx, H_xy], [H_xy, H_yy]])
            evals = np.linalg.eigvalsh(H)
            lambda_1[i, j] = evals[0]
            lambda_2[i, j] = evals[1]
    
    det_H = lambda_1 * lambda_2
    return det_J_grid, grad_norm, det_H, lambda_1, lambda_2

In [ ]:
# Hyperparameters for Visualization 2
field_type_2 = 'double_gyre'  # 'double_vortex', 'double_saddle', 'double_gyre', 'four_vortex'
eps_1 = 0.3                   # gradient tolerance
eps_2 = 0.01                  # degeneracy floor for |det(H)|

# Domain selection
if field_type_2 == 'double_gyre':
    x_range, y_range = [0, 2], [0, 1]
    grid_res = 80
else:
    x_range, y_range = [-5, 5], [-5, 5]
    grid_res = 80

field = VectorField(field_type=field_type_2)

xs = np.linspace(x_range[0], x_range[1], grid_res)
ys = np.linspace(y_range[0], y_range[1], grid_res)
X, Y = np.meshgrid(xs, ys)

det_J_grid, grad_norm, det_H, lambda_1, lambda_2 = compute_scalar_field_analysis(
    field, x_range, y_range, grid_res=grid_res)

degenerate_mask, saddle_mask, minmax_mask = classify_critical_points(
    grad_norm, det_H, lambda_1, lambda_2, eps_1, eps_2)

# Compute vector field for right panel
U = np.zeros_like(X)
V = np.zeros_like(X)
for i in range(grid_res):
    for j in range(grid_res):
        u, v = field.evaluate(X[i, j], Y[i, j])
        U[i, j], V[i, j] = u, v

# Create figure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'{field_type_2} | ε₁={eps_1:.3f}, ε₂={eps_2:.5f}',
             fontsize=12, fontweight='bold')

# LEFT: det(J) scalar field with critical points
ax = axes[0]
cs = ax.contourf(X, Y, det_J_grid, levels=20, cmap='RdBu_r', alpha=0.8)
ax.contour(X, Y, det_J_grid, levels=[0], colors='black', linewidths=2)
cbar = plt.colorbar(cs, ax=ax)
cbar.set_label('det(J)')

# Plot critical points as X's
marker_size = 80
degenerate_pts = np.where(degenerate_mask)
saddle_pts = np.where(saddle_mask)
minmax_pts = np.where(minmax_mask)

if len(degenerate_pts[0]) > 0:
    ax.scatter(X[degenerate_pts], Y[degenerate_pts], marker='x', s=marker_size,
               c='cyan', linewidths=2, label=f'Degenerate ({len(degenerate_pts[0])})')
if len(saddle_pts[0]) > 0:
    ax.scatter(X[saddle_pts], Y[saddle_pts], marker='x', s=marker_size,
               c='red', linewidths=2, label=f'Saddle ({len(saddle_pts[0])})')
if len(minmax_pts[0]) > 0:
    ax.scatter(X[minmax_pts], Y[minmax_pts], marker='x', s=marker_size,
               c='green', linewidths=2, label=f'Min/Max ({len(minmax_pts[0])})')

ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('det(J) Scalar Field with Critical Points')
ax.set_aspect('equal')
ax.legend(loc='upper right', fontsize=9)

# RIGHT: Vector field flow
ax = axes[1]
speed = np.sqrt(U**2 + V**2)
lw = 2 * speed / (speed.max() + 1e-10)
ax.streamplot(X, Y, U, V, color=speed, cmap='viridis', linewidth=lw, density=1.5)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Vector Field Flow')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

# Visualization 3: Comprehensive Field Analysis (4-Panel)

In [ ]:
def compute_critical_point_mask(det_grid, grad_x, grad_y, lambda_max, lambda_min,
                                 eps_g, eps_lambda):
    """Compute boolean mask for index-1 saddle detection."""
    grad_norm = np.sqrt(grad_x**2 + grad_y**2)
    mask = (grad_norm < eps_g) & (lambda_max > eps_lambda) & (lambda_min < -eps_lambda)
    return mask


def compute_det_field_and_derivatives(field, x_range, y_range, grid_res=100):
    """Compute det(J), ∇det(J), and ∇²det(J) on a grid."""
    xs = np.linspace(x_range[0], x_range[1], grid_res)
    ys = np.linspace(y_range[0], y_range[1], grid_res)
    X, Y = np.meshgrid(xs, ys)
    
    det_grid = np.zeros_like(X)
    grad_x = np.zeros_like(X)
    grad_y = np.zeros_like(X)
    lambda_max = np.zeros_like(X)
    lambda_min = np.zeros_like(X)
    
    h = (x_range[1] - x_range[0]) / (grid_res - 1) * 2
    
    for i in range(grid_res):
        for j in range(grid_res):
            x, y = X[i, j], Y[i, j]
            
            J = field.jacobian_twosided(x, y, h=h/2)
            det_grid[i, j] = np.linalg.det(J)
            
            J_xp = field.jacobian_twosided(x + h, y, h=h/2)
            J_xm = field.jacobian_twosided(x - h, y, h=h/2)
            J_yp = field.jacobian_twosided(x, y + h, h=h/2)
            J_ym = field.jacobian_twosided(x, y - h, h=h/2)
            
            grad_x[i, j] = (np.linalg.det(J_xp) - np.linalg.det(J_xm)) / (2*h)
            grad_y[i, j] = (np.linalg.det(J_yp) - np.linalg.det(J_ym)) / (2*h)
            
            ddet_xp_yp = np.linalg.det(field.jacobian_twosided(x + h, y + h, h=h/2))
            ddet_xp_ym = np.linalg.det(field.jacobian_twosided(x + h, y - h, h=h/2))
            ddet_xm_yp = np.linalg.det(field.jacobian_twosided(x - h, y + h, h=h/2))
            ddet_xm_ym = np.linalg.det(field.jacobian_twosided(x - h, y - h, h=h/2))
            
            H_xx = (ddet_xp_yp - 2*det_grid[i, j] + ddet_xm_ym) / (h**2)
            H_yy = (ddet_xp_yp - 2*det_grid[i, j] + ddet_xm_ym) / (h**2)
            H_xy = (ddet_xp_yp - ddet_xp_ym - ddet_xm_yp + ddet_xm_ym) / (4*h**2)
            
            H = np.array([[H_xx, H_xy], [H_xy, H_yy]])
            evals = np.linalg.eigvalsh(H)
            lambda_max[i, j] = evals[1]
            lambda_min[i, j] = evals[0]
    
    return det_grid, grad_x, grad_y, lambda_max, lambda_min

In [ ]:
# Hyperparameters for Visualization 3
field_type_3 = 'double_vortex'  # 'double_vortex', 'double_saddle', 'double_gyre', 'four_vortex'
eps_g_3 = 0.5                   # gradient tolerance
eps_lambda_3 = 0.5              # eigenvalue gap

# Domain selection
if field_type_3 == 'double_gyre':
    x_range, y_range = [0, 2], [0, 1]
    grid_res = 80
else:
    x_range, y_range = [-5, 5], [-5, 5]
    grid_res = 100

field = VectorField(field_type=field_type_3)

xs = np.linspace(x_range[0], x_range[1], grid_res)
ys = np.linspace(y_range[0], y_range[1], grid_res)
X, Y = np.meshgrid(xs, ys)

# Vector field
U = np.zeros_like(X)
V = np.zeros_like(X)
for i in range(grid_res):
    for j in range(grid_res):
        u, v = field.evaluate(X[i, j], Y[i, j])
        U[i, j], V[i, j] = u, v

# Determinant and eigenvalue fields
det_grid, grad_x, grad_y, lambda_max, lambda_min = compute_det_field_and_derivatives(
    field, x_range, y_range, grid_res=grid_res)

# Critical point mask
cp_mask = compute_critical_point_mask(det_grid, grad_x, grad_y, lambda_max, lambda_min,
                                       eps_g_3, eps_lambda_3)

# Create figure
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle(f'Field Analysis: {field_type_3} | ε_g={eps_g_3:.3f}, ε_λ={eps_lambda_3:.3f}',
             fontsize=14, fontweight='bold')

# 1. Vector field with streamlines
ax = axes[0, 0]
speed = np.sqrt(U**2 + V**2)
lw = 2 * speed / speed.max()
ax.streamplot(X, Y, U, V, color=speed, cmap='viridis', linewidth=lw, density=1.5)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Vector Field')
ax.set_aspect('equal')

# 2. det(J) scalar field
ax = axes[0, 1]
cs = ax.contourf(X, Y, det_grid, levels=20, cmap='RdBu_r')
ax.contour(X, Y, det_grid, levels=[0], colors='black', linewidths=2)
cbar = plt.colorbar(cs, ax=ax)
cbar.set_label('det(J)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Determinant of Jacobian')
ax.set_aspect('equal')

# 3. Critical point region
ax = axes[1, 0]
ax.contourf(X, Y, cp_mask.astype(float), levels=[0.5, 1.5], colors=['lightgreen'], alpha=0.7)
ax.contour(X, Y, det_grid, levels=[0], colors='gray', linewidths=1, linestyles='--')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(f'Critical Point Region\n(‖g‖ < {eps_g_3:.3f} AND λ₊ > {eps_lambda_3:.3f} AND λ₋ < {-eps_lambda_3:.3f})')
ax.set_aspect('equal')
num_points = np.sum(cp_mask)
ax.text(0.02, 0.98, f'{num_points} grid points satisfy criterion',
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# 4. Eigenvalue landscape
ax = axes[1, 1]
grad_norm = np.sqrt(grad_x**2 + grad_y**2)
cs = ax.contourf(X, Y, grad_norm, levels=20, cmap='plasma')
cbar = plt.colorbar(cs, ax=ax)
cbar.set_label('‖∇det(J)‖')
ax.contour(X, Y, grad_norm, levels=[eps_g_3], colors='cyan', linewidths=2, linestyles='-')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title(f'Gradient Norm (cyan: ‖g‖ = ε_g = {eps_g_3:.3f})')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()